# 09b — Create Governed Pages (Business Glossary)

Creates **5 governed Pages** (business-term definitions) that Genie One reads
as part of its ontology — grounding natural-language answers in official
definitions.

Pages are attached to the domains/subdomains created in `09a_create_domains`.

> **NOTE:** The Pages REST API (`/api/2.0/pages`) may not be available on all
> workspaces. This notebook attempts programmatic creation and falls back
> gracefully. If the API is not exposed, create the pages manually using the
> instructions at the bottom of this notebook.

> **Pre-requisites:** `09a_create_domains` must have run first (governed tags,
> domains, subdomains, and asset tagging must be in place).

In [0]:
%run ../00_config

In [0]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()
NS = CFG.namespace


def _do(method, path, body=None, params=None):
    """Call the Databricks REST API via the SDK's authenticated client."""
    headers = {"Accept": "application/json", "Content-Type": "application/json"}
    if w.config.workspace_id:
        headers["X-Databricks-Workspace-Id"] = w.config.workspace_id
    return w.api_client.do(method, path, body=body, query=params, headers=headers)


print(f"Workspace: {w.config.host}")
print(f"Namespace: {NS}")

In [0]:
# Read domain IDs persisted by 09a_create_domains.
domains = {r["domain_key"]: r["domain_id"] for r in spark.table(CFG.fq("_DOMAINS")).collect()}

PARENT_TAG = "SAP_Purchasing"
SUB_TAG_1 = f"{PARENT_TAG}/Sourcing_Supplier_Risk"
SUB_TAG_2 = f"{PARENT_TAG}/Purchase_Order_Analytics"

domain_id = domains.get(PARENT_TAG)
sub1_id = domains.get(SUB_TAG_1)
sub2_id = domains.get(SUB_TAG_2)

print(f"Parent domain:  {domain_id}")
print(f"Subdomain 1:    {sub1_id}")
print(f"Subdomain 2:    {sub2_id}")

## Page definitions & programmatic creation
The 5 pages below are business-term definitions that Genie One reads as ontology.
Each page is attached to a domain or subdomain and carries synonyms so Genie can
match natural-language questions to the right concept.

The separation principle:
* **Comments** = what a column/table IS (physical)
* **Pages** = what a business concept MEANS (definitional) + WHY it matters
* **Metric Views** = HOW a KPI is calculated
* **Functions** = complex multi-table trusted queries

In [0]:
# ---------------------------------------------------------------------------
# Programmatic Page creation via the Pages REST API.
# The API may not be available on all workspaces. This code discovers the
# endpoint and attempts creation; if the API is not exposed, it falls back
# gracefully and reports which pages were skipped.
# ---------------------------------------------------------------------------

# ---- Page definitions --------------------------------------------------------
PAGE_DEFS = [
    {
        "name": "Multi-Sourcing Strategy",
        "description": "What multi-sourcing means and why it matters for supply continuity.",
        "synonyms": ["multi-sourcing", "single-sourced", "dual sourcing", "second source"],
        "domain_id": sub1_id,
        "related_assets": [f"{NS}.mv_sourcing_coverage", f"{NS}.sourcing"],
        "body": (
            "## Definition\n\n"
            "A material is **multi-sourced** when it has **\u22652 valid, non-blocked vendors** "
            "(IS_VALID_SOURCE = true) in its source list at the relevant plant. "
            "Materials below this threshold are flagged as **single-source risks**.\n\n"
            "## Why it matters\n\n"
            "Single-sourced materials create supply-chain concentration risk: if the sole "
            "vendor experiences disruption, the production line has no fallback. "
            "Multi-sourcing is the primary mitigation.\n\n"
            "## Threshold\n\n"
            "The standard threshold is **2**. The certified function "
            "`materials_without_multisourcing(min_vendors)` is parameterized.\n\n"
            "## Related assets\n\n"
            "* **Metric view:** `mv_sourcing_coverage` \u2014 vendor count by material, plant, group\n"
            "* **Function:** `materials_without_multisourcing(min_vendors)` \u2014 Q1\n"
            "* **Table:** `SOURCING` (IS_VALID_SOURCE column)"
        ),
    },
    {
        "name": "Delivery Risk",
        "description": "How delivery risk is defined, assessed, and why it is time-sliced.",
        "synonyms": ["delivery risk", "supplier risk", "risk flag", "risk score", "risk window"],
        "domain_id": sub1_id,
        "related_assets": [f"{NS}.mv_supplier_risk", f"{NS}.supplier_risk"],
        "body": (
            "## Definition\n\n"
            "Delivery risk is a **time-sliced** assessment. A supplier carries **active** "
            "delivery risk when:\n\n"
            "1. `DELIVERY_RISK_FLAG = 'X'`, **and**\n"
            "2. The reference date falls **between VALID_FROM and VALID_TO**.\n\n"
            "Risk outside this window is historical, not current.\n\n"
            "## Risk dimensions\n\n"
            "Each risk record carries:\n"
            "* **RISK_CATEGORY** \u2014 financial, quality, geopolitical, or logistics\n"
            "* **RISK_SCORE** \u2014 numeric 0\u2013100 (higher = more severe)\n"
            "* **DELIVERY_RISK_FLAG** \u2014 'X' if the supplier is flagged\n\n"
            "## Why it matters\n\n"
            "A material that is **single-sourced AND its sole supplier has active delivery "
            "risk** is the highest-priority supply-continuity exposure.\n\n"
            "## Related assets\n\n"
            "* **Metric view:** `mv_supplier_risk` \u2014 risk exposure by category/flag/country\n"
            "* **Function:** `single_sourced_risky(as_of_date)` \u2014 Q3\n"
            "* **Table:** `SUPPLIER_RISK` (VALID_FROM, VALID_TO columns)"
        ),
    },
    {
        "name": "On-Time Delivery (OTD)",
        "description": "How OTD rate is calculated and what counts as on-time.",
        "synonyms": ["OTD", "on-time delivery", "on-time rate", "delivery performance"],
        "domain_id": sub2_id,
        "related_assets": [f"{NS}.mv_delivery_performance", f"{NS}.purchase_order"],
        "body": (
            "## Definition\n\n"
            "**OTD rate** = on-time lines \u00f7 received lines.\n\n"
            "A PO line is **on-time** when `GR_DATE \u2264 REQUESTED_DATE` (goods received by "
            "the requested date). Lines where `GR_DATE IS NULL` (not yet received) are "
            "**excluded** from both numerator and denominator.\n\n"
            "## Standard window\n\n"
            "The default analysis window is the **last 12 months** of goods-receipt dates.\n\n"
            "## Why it matters\n\n"
            "OTD is the primary measure of supplier delivery performance. Suppliers with "
            "persistently low OTD rates are candidates for escalation, re-sourcing, or "
            "additional risk mitigation.\n\n"
            "## Related assets\n\n"
            "* **Metric view:** `mv_delivery_performance` \u2014 OTD % by supplier/plant/month\n"
            "* **Table:** `PURCHASE_ORDER` (ON_TIME_FLAG, GR_DATE columns)"
        ),
    },
    {
        "name": "Quota Arrangement",
        "description": "How procurement volume is split across approved vendors.",
        "synonyms": ["quota", "quota split", "quota concentration", "vendor share"],
        "domain_id": sub1_id,
        "related_assets": [f"{NS}.sourcing"],
        "body": (
            "## Definition\n\n"
            "A **quota arrangement** splits procurement of a material across approved "
            "vendors by percentage. The `QUOTA` column in the SOURCING table represents "
            "the vendor's allocated share.\n\n"
            "## NULL quota\n\n"
            "A `NULL` quota means the vendor is **listed as a valid source** but has "
            "**no formal quota split** assigned. This is a meaningful gap, not a data "
            "error \u2014 it indicates the source list is approved but procurement volume "
            "allocation is not governed.\n\n"
            "## Concentration risk\n\n"
            "When a single vendor holds **>80%** of the quota for a material, it represents "
            "a concentration risk even if the material is technically multi-sourced. "
            "The function `quota_concentration(threshold_pct)` flags these cases.\n\n"
            "## Related assets\n\n"
            "* **Function:** `quota_concentration(threshold_pct)` \u2014 Q6\n"
            "* **Table:** `SOURCING` (QUOTA column)"
        ),
    },
    {
        "name": "SAP Code Glossary",
        "description": "How cryptic SAP codes are translated to human-readable labels.",
        "synonyms": ["SAP codes", "value dictionary", "code mapping", "MTART", "SPERM", "NOTKZ"],
        "domain_id": domain_id,
        "related_assets": [f"{NS}.value_dictionary"],
        "body": (
            "## Definition\n\n"
            "SAP uses **short alphanumeric codes** throughout its modules. The "
            "`VALUE_DICTIONARY` table maps each code to a human-readable label so that "
            "Genie (and humans) can translate between plain English and SAP terminology.\n\n"
            "## Key domains\n\n"
            "| Code domain | Example code | Meaning |\n"
            "| --- | --- | --- |\n"
            "| MTART (material type) | ROH | Raw material |\n"
            "| MATKL (material group) | 001 | Mechanical components |\n"
            "| SPERM (purchasing block) | X | Supplier is blocked for purchasing |\n"
            "| NOTKZ (source-list block) | X | Source record is blocked |\n"
            "| DELIVERY_RISK_FLAG | X | Active delivery risk |\n"
            "| LAND1 (country) | DE | Germany |\n\n"
            "## How Genie uses it\n\n"
            "When a user asks about 'blocked suppliers', Genie translates this to "
            "`SPERM = 'X'` using the value dictionary. When a user asks about "
            "'raw materials', Genie translates to `MTART = 'ROH'`.\n\n"
            "## Related assets\n\n"
            "* **Table:** `VALUE_DICTIONARY` (DOMAIN, CODE, LABEL, DESCRIPTION columns)"
        ),
    },
]


# ---- API discovery -----------------------------------------------------------
PAGES_API = None
_NOT_FOUND_MARKERS = ("Not Found", "No API found", "404")

for path in ["/api/2.0/pages", "/api/2.1/pages",
             "/api/2.0/discover/pages", "/api/2.1/discover/pages"]:
    try:
        _do("GET", path)
        PAGES_API = path
        break
    except Exception as e:
        if not any(m in str(e) for m in _NOT_FOUND_MARKERS):
            PAGES_API = path
            break

if PAGES_API:
    print(f"Pages API found: {PAGES_API}")
else:
    print("Pages API not available on this workspace.")
    print("Pages must be created manually (see instructions below).")


# ---- Page creation -----------------------------------------------------------
def find_page(name):
    """Return the page ID for a given name, or None."""
    if not PAGES_API:
        return None
    try:
        resp = _do("GET", PAGES_API)
        for p in resp.get("pages", []):
            if (p.get("name") or "") == name:
                return p.get("page_id") or p.get("id")
    except Exception:
        pass
    return None


def create_page(page_def):
    """Create or overwrite a governed Page via the REST API. Returns True on success.

    If a page with the same name already exists, it is deleted first so that
    the latest definition always takes effect (idempotent overwrite).
    """
    if not PAGES_API:
        print(f"  \u2717 API not available \u2014 skipped: {page_def['name']}")
        return False
    # Overwrite: delete existing page first.
    existing_id = find_page(page_def["name"])
    if existing_id:
        try:
            _do("DELETE", f"{PAGES_API}/{existing_id}")
            print(f"  \u2022 Deleted existing page: {page_def['name']} (id={existing_id})")
        except Exception as e:
            print(f"  \u2022 Could not delete existing page {page_def['name']}: {e}")
    body = {
        "name": page_def["name"],
        "description": page_def["description"],
        "body": page_def["body"],
        "synonyms": page_def.get("synonyms", []),
    }
    if page_def.get("domain_id"):
        body["domain_id"] = page_def["domain_id"]
    if page_def.get("related_assets"):
        body["related_assets"] = page_def["related_assets"]
    try:
        resp = _do("POST", PAGES_API, body=body)
        page_id = resp.get("page_id") or resp.get("id")
        print(f"  \u2713 Created: {page_def['name']} (id={page_id})")
        return True
    except Exception as e:
        if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
            print(f"  \u2022 Already exists: {page_def['name']}")
            return True
        print(f"  \u2717 Failed: {page_def['name']} \u2014 {e}")
        return False


print("\n--- Creating governed Pages ---")
created = sum(create_page(p) for p in PAGE_DEFS)
print(f"\nPages created: {created}/{len(PAGE_DEFS)}")

if created < len(PAGE_DEFS):
    print("\n\u26a0  Some pages could not be created programmatically.")
    print("  See the manual creation instructions below.")

dbutils.notebook.exit(
    f"09b_create_pages: {'SUCCESS' if created == len(PAGE_DEFS) else 'PARTIAL'}"
    f" \u2014 {created}/{len(PAGE_DEFS)} pages created."
)

---
# Manual Page Creation (if the Pages API is not available)

If the Pages REST API is not yet exposed on your workspace, create the 5 pages
manually using **one of the two methods** below.

---

## Method A — Create via the Discover UI

1. Open **Discover** (left sidebar → Discover).
2. Click the target **domain** or **subdomain** (see each page below for which one).
3. Click **"+ New page"** (top-right within the domain).
4. Fill in the **Name**, **Description**, **Synonyms**, and **Body** exactly as
   shown in each page section below.
5. Optionally attach **Related assets** (tables, metric views, functions).
6. Click **Save** (this creates the page as a **draft**).

---

## Method B — Create via Genie Code prompt

1. Open the **Discover** page in your workspace.
2. Open **Genie Code** (the assistant sidebar).
3. Copy the prompt from each page section below and paste it into Genie Code.
4. Genie Code will create the page for you (as a **draft**).

---

## Publishing a Page (draft → production)

Pages are created in **draft** state (visible only to curators / owners). To make
a page visible to all users:

1. Open **Discover** → navigate to the domain/subdomain containing the page.
2. Click on the **page name** to open it.
3. In the page editor, click the **"Publish"** button (top-right).
4. Confirm the publish action.
5. The page is now **live** and visible to all workspace users. Genie One will
   read it as part of its ontology.

> **Tip:** You can edit a published page at any time. Edits create a new draft
> that must be re-published to take effect.

---
## Page 1 — Multi-Sourcing Strategy

| Field | Value |
| --- | --- |
| **Attach to** | Sourcing & Supplier Risk (subdomain) |
| **Description** | What multi-sourcing means and why it matters for supply continuity. |
| **Synonyms** | multi-sourcing, single-sourced, dual sourcing, second source |
| **Related assets** | `mv_sourcing_coverage`, `SOURCING` |

### Body (paste into the page editor or Genie Code prompt)

> **Definition** — A material is **multi-sourced** when it has ≥2 valid,
> non-blocked vendors (IS_VALID_SOURCE = true) in its source list at the relevant
> plant. Materials below this threshold are flagged as **single-source risks**.
>
> **Why it matters** — Single-sourced materials create supply-chain concentration
> risk: if the sole vendor experiences disruption, the production line has no
> fallback. Multi-sourcing is the primary mitigation.
>
> **Threshold** — The standard threshold is **2**. The certified function
> `materials_without_multisourcing(min_vendors)` is parameterized so the
> threshold can be adjusted.
>
> **Related assets:**
> * **Metric view:** `mv_sourcing_coverage` — vendor count by material, plant, group
> * **Function:** `materials_without_multisourcing(min_vendors)` — Q1
> * **Table:** `SOURCING` (IS_VALID_SOURCE column)

### Genie Code prompt

````text
Create a governed Page called "Multi-Sourcing Strategy" in the "Sourcing & Supplier Risk" subdomain (tag_key=SAP_Purchasing/Sourcing_Supplier_Risk).
Description: What multi-sourcing means and why it matters for supply continuity.
Synonyms: multi-sourcing, single-sourced, dual sourcing, second source
Related assets: max_kohler_demo_catalog.purchasing_demo.mv_sourcing_coverage, max_kohler_demo_catalog.purchasing_demo.sourcing
Body:
## Definition
A material is **multi-sourced** when it has ≥2 valid, non-blocked vendors (IS_VALID_SOURCE = true) in its source list at the relevant plant. Materials below this threshold are flagged as **single-source risks**.
## Why it matters
Single-sourced materials create supply-chain concentration risk: if the sole vendor experiences disruption, the production line has no fallback. Multi-sourcing is the primary mitigation.
## Threshold
The standard threshold is **2**. The certified function `materials_without_multisourcing(min_vendors)` is parameterized so the threshold can be adjusted.
## Related assets
* **Metric view:** `mv_sourcing_coverage` — vendor count by material, plant, group
* **Function:** `materials_without_multisourcing(min_vendors)` — Q1
* **Table:** `SOURCING` (IS_VALID_SOURCE column)
````

---
## Page 2 — Delivery Risk

| Field | Value |
| --- | --- |
| **Attach to** | Sourcing & Supplier Risk (subdomain) |
| **Description** | How delivery risk is defined, assessed, and why it is time-sliced. |
| **Synonyms** | delivery risk, supplier risk, risk flag, risk score, risk window |
| **Related assets** | `mv_supplier_risk`, `SUPPLIER_RISK` |

### Body (paste into the page editor or Genie Code prompt)

> **Definition** — Delivery risk is a **time-sliced** assessment. A supplier
> carries **active** delivery risk when:
> 1. `DELIVERY_RISK_FLAG = 'X'`, **and**
> 2. The reference date falls **between VALID_FROM and VALID_TO**.
>
> Risk outside this window is historical, not current.
>
> **Risk dimensions** — Each risk record carries:
> * **RISK_CATEGORY** — financial, quality, geopolitical, or logistics
> * **RISK_SCORE** — numeric 0–100 (higher = more severe)
> * **DELIVERY_RISK_FLAG** — 'X' if the supplier is flagged
>
> **Why it matters** — A material that is **single-sourced AND its sole supplier
> has active delivery risk** is the highest-priority supply-continuity exposure.
>
> **Related assets:**
> * **Metric view:** `mv_supplier_risk` — risk exposure by category/flag/country
> * **Function:** `single_sourced_risky(as_of_date)` — Q3
> * **Table:** `SUPPLIER_RISK` (VALID_FROM, VALID_TO columns)

### Genie Code prompt

````text
Create a governed Page called "Delivery Risk" in the "Sourcing & Supplier Risk" subdomain (tag_key=SAP_Purchasing/Sourcing_Supplier_Risk).
Description: How delivery risk is defined, assessed, and why it is time-sliced.
Synonyms: delivery risk, supplier risk, risk flag, risk score, risk window
Related assets: max_kohler_demo_catalog.purchasing_demo.mv_supplier_risk, max_kohler_demo_catalog.purchasing_demo.supplier_risk
Body:
## Definition
Delivery risk is a **time-sliced** assessment. A supplier carries **active** delivery risk when:
1. `DELIVERY_RISK_FLAG = 'X'`, **and**
2. The reference date falls **between VALID_FROM and VALID_TO**.
Risk outside this window is historical, not current.
## Risk dimensions
Each risk record carries:
* **RISK_CATEGORY** — financial, quality, geopolitical, or logistics
* **RISK_SCORE** — numeric 0–100 (higher = more severe)
* **DELIVERY_RISK_FLAG** — 'X' if the supplier is flagged
## Why it matters
A material that is **single-sourced AND its sole supplier has active delivery risk** is the highest-priority supply-continuity exposure.
## Related assets
* **Metric view:** `mv_supplier_risk` — risk exposure by category/flag/country
* **Function:** `single_sourced_risky(as_of_date)` — Q3
* **Table:** `SUPPLIER_RISK` (VALID_FROM, VALID_TO columns)
````

---
## Page 3 — On-Time Delivery (OTD)

| Field | Value |
| --- | --- |
| **Attach to** | Purchase Order Analytics (subdomain) |
| **Description** | How OTD rate is calculated and what counts as on-time. |
| **Synonyms** | OTD, on-time delivery, on-time rate, delivery performance |
| **Related assets** | `mv_delivery_performance`, `PURCHASE_ORDER` |

### Body (paste into the page editor or Genie Code prompt)

> **Definition** — **OTD rate** = on-time lines ÷ received lines. A PO line is
> **on-time** when `GR_DATE ≤ REQUESTED_DATE` (goods received by the requested
> date). Lines where `GR_DATE IS NULL` (not yet received) are **excluded** from
> both numerator and denominator.
>
> **Standard window** — The default analysis window is the **last 12 months** of
> goods-receipt dates.
>
> **Why it matters** — OTD is the primary measure of supplier delivery
> performance. Suppliers with persistently low OTD rates are candidates for
> escalation, re-sourcing, or additional risk mitigation.
>
> **Related assets:**
> * **Metric view:** `mv_delivery_performance` — OTD % by supplier/plant/month
> * **Table:** `PURCHASE_ORDER` (ON_TIME_FLAG, GR_DATE columns)

### Genie Code prompt

````text
Create a governed Page called "On-Time Delivery (OTD)" in the "Purchase Order Analytics" subdomain (tag_key=SAP_Purchasing/Purchase_Order_Analytics).
Description: How OTD rate is calculated and what counts as on-time.
Synonyms: OTD, on-time delivery, on-time rate, delivery performance
Related assets: max_kohler_demo_catalog.purchasing_demo.mv_delivery_performance, max_kohler_demo_catalog.purchasing_demo.purchase_order
Body:
## Definition
**OTD rate** = on-time lines ÷ received lines. A PO line is **on-time** when `GR_DATE ≤ REQUESTED_DATE` (goods received by the requested date). Lines where `GR_DATE IS NULL` (not yet received) are **excluded** from both numerator and denominator.
## Standard window
The default analysis window is the **last 12 months** of goods-receipt dates.
## Why it matters
OTD is the primary measure of supplier delivery performance. Suppliers with persistently low OTD rates are candidates for escalation, re-sourcing, or additional risk mitigation.
## Related assets
* **Metric view:** `mv_delivery_performance` — OTD % by supplier/plant/month
* **Table:** `PURCHASE_ORDER` (ON_TIME_FLAG, GR_DATE columns)
````

---
## Page 4 — Quota Arrangement

| Field | Value |
| --- | --- |
| **Attach to** | Sourcing & Supplier Risk (subdomain) |
| **Description** | How procurement volume is split across approved vendors. |
| **Synonyms** | quota, quota split, quota concentration, vendor share |
| **Related assets** | `SOURCING` |

### Body (paste into the page editor or Genie Code prompt)

> **Definition** — A **quota arrangement** splits procurement of a material across
> approved vendors by percentage. The `QUOTA` column in the SOURCING table
> represents the vendor's allocated share.
>
> **NULL quota** — A `NULL` quota means the vendor is **listed as a valid source**
> but has **no formal quota split** assigned. This is a meaningful gap, not a data
> error — it indicates the source list is approved but procurement volume
> allocation is not governed.
>
> **Concentration risk** — When a single vendor holds **>80%** of the quota for a
> material, it represents a concentration risk even if the material is technically
> multi-sourced. The function `quota_concentration(threshold_pct)` flags these
> cases.
>
> **Related assets:**
> * **Function:** `quota_concentration(threshold_pct)` — Q6
> * **Table:** `SOURCING` (QUOTA column)

### Genie Code prompt

````text
Create a governed Page called "Quota Arrangement" in the "Sourcing & Supplier Risk" subdomain (tag_key=SAP_Purchasing/Sourcing_Supplier_Risk).
Description: How procurement volume is split across approved vendors.
Synonyms: quota, quota split, quota concentration, vendor share
Related assets: max_kohler_demo_catalog.purchasing_demo.sourcing
Body:
## Definition
A **quota arrangement** splits procurement of a material across approved vendors by percentage. The `QUOTA` column in the SOURCING table represents the vendor's allocated share.
## NULL quota
A `NULL` quota means the vendor is **listed as a valid source** but has **no formal quota split** assigned. This is a meaningful gap, not a data error — it indicates the source list is approved but procurement volume allocation is not governed.
## Concentration risk
When a single vendor holds **>80%** of the quota for a material, it represents a concentration risk even if the material is technically multi-sourced. The function `quota_concentration(threshold_pct)` flags these cases.
## Related assets
* **Function:** `quota_concentration(threshold_pct)` — Q6
* **Table:** `SOURCING` (QUOTA column)
````

---
## Page 5 — SAP Code Glossary

| Field | Value |
| --- | --- |
| **Attach to** | SAP Purchasing **parent domain** (not a subdomain) |
| **Description** | How cryptic SAP codes are translated to human-readable labels. |
| **Synonyms** | SAP codes, value dictionary, code mapping, MTART, SPERM, NOTKZ |
| **Related assets** | `VALUE_DICTIONARY` |

### Body (paste into the page editor or Genie Code prompt)

> **Definition** — SAP uses **short alphanumeric codes** throughout its modules.
> The `VALUE_DICTIONARY` table maps each code to a human-readable label so that
> Genie (and humans) can translate between plain English and SAP terminology.
>
> **Key domains:**
>
> | Code domain | Example code | Meaning |
> | --- | --- | --- |
> | MTART (material type) | ROH | Raw material |
> | MATKL (material group) | 001 | Mechanical components |
> | SPERM (purchasing block) | X | Supplier is blocked for purchasing |
> | NOTKZ (source-list block) | X | Source record is blocked |
> | DELIVERY_RISK_FLAG | X | Active delivery risk |
> | LAND1 (country) | DE | Germany |
>
> **How Genie uses it** — When a user asks about 'blocked suppliers', Genie
> translates this to `SPERM = 'X'` using the value dictionary. When a user asks
> about 'raw materials', Genie translates to `MTART = 'ROH'`.
>
> **Related assets:**
> * **Table:** `VALUE_DICTIONARY` (DOMAIN, CODE, LABEL, DESCRIPTION columns)

### Genie Code prompt

````text
Create a governed Page called "SAP Code Glossary" in the "SAP Purchasing" PARENT domain (tag_key=SAP_Purchasing). NOT a subdomain.
Description: How cryptic SAP codes are translated to human-readable labels.
Synonyms: SAP codes, value dictionary, code mapping, MTART, SPERM, NOTKZ
Related assets: max_kohler_demo_catalog.purchasing_demo.value_dictionary
Body:
## Definition
SAP uses **short alphanumeric codes** throughout its modules. The `VALUE_DICTIONARY` table maps each code to a human-readable label so that Genie (and humans) can translate between plain English and SAP terminology.
## Key domains
| Code domain | Example code | Meaning |
| --- | --- | --- |
| MTART (material type) | ROH | Raw material |
| MATKL (material group) | 001 | Mechanical components |
| SPERM (purchasing block) | X | Supplier is blocked for purchasing |
| NOTKZ (source-list block) | X | Source record is blocked |
| DELIVERY_RISK_FLAG | X | Active delivery risk |
| LAND1 (country) | DE | Germany |
## How Genie uses it
When a user asks about 'blocked suppliers', Genie translates this to `SPERM = 'X'` using the value dictionary. When a user asks about 'raw materials', Genie translates to `MTART = 'ROH'`.
## Related assets
* **Table:** `VALUE_DICTIONARY` (DOMAIN, CODE, LABEL, DESCRIPTION columns)
````